<a href="https://colab.research.google.com/github/shankar27git/trainable-Quantum-Kernel/blob/main/trainable_Quantum_Kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q medmnist pennylane torch torchvision scikit-learn matplotlib


In [ ]:
import torch
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

import pennylane as qml


AssertionError: record_shapeenv_event should only wrap methods on ShapeEnv; refactor your code so that it calls into a method on ShapeEnv

In [ ]:
import torch
import torchvision.transforms as transforms
import pennylane as qml
import medmnist

print("torch:", torch.__version__)
print("torchvision ok")
print("pennylane:", qml.__version__)
print("medmnist:", medmnist.__version__)


AssertionError: record_shapeenv_event should only wrap methods on ShapeEnv; refactor your code so that it calls into a method on ShapeEnv

In [ ]:
# ==============================
# Imports and Setup
# ==============================

import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

import medmnist
from medmnist import INFO

import pennylane as qml
from pennylane import numpy as pnp

import matplotlib.pyplot as plt

# ==============================
# Reproducibility
# ==============================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pnp.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==============================
# Device
# ==============================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("Torch:", torch.__version__, "| TorchVision:", transforms.__module__)


AssertionError: record_shapeenv_event should only wrap methods on ShapeEnv; refactor your code so that it calls into a method on ShapeEnv

In [ ]:
# Load MedMNIST Dataset (PathMNIST)
DATA_FLAG = "pathmnist"  # Can also use "bloodmnist", etc.

info = INFO[DATA_FLAG]
n_channels = info["n_channels"]
n_classes = len(info["label"])
Task = getattr(medmnist, info["python_class"])

transform = transforms.Compose([
    transforms.ToTensor(),  # outputs [C,H,W] in [0,1]
])

train_dataset = Task(split="train", transform=transform, download=True)
test_dataset  = Task(split="test",  transform=transform, download=True)

print(f"Dataset: {DATA_FLAG}")
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Channels: {n_channels}, Classes: {n_classes}")

100%|██████████| 206M/206M [00:08<00:00, 24.3MB/s]


Dataset: pathmnist
Train size: 89996
Test size: 7180
Channels: 3, Classes: 9


In [ ]:
# CNN Feature Extractor
class SmallCNN(nn.Module):
    def __init__(self, in_channels=3, feature_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 14x14
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 7x7
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)   # 1x1
        )
        self.fc = nn.Linear(64, feature_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

def get_loader(dataset, batch_size, subset_fraction=None, shuffle=True):
    if subset_fraction is not None:
        n = len(dataset)
        subset_size = int(n * subset_fraction)
        indices = list(range(n))
        np.random.shuffle(indices)
        subset_indices = indices[:subset_size]
        subset = torch.utils.data.Subset(dataset, subset_indices)
        return DataLoader(subset, batch_size=batch_size, shuffle=shuffle, drop_last=False)
    else:
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

In [ ]:
# CNN Training and Feature Extraction
def train_cnn_feature_extractor(train_loader, feature_dim=64, epochs=5, lr=1e-3):
    model = SmallCNN(in_channels=n_channels, feature_dim=feature_dim).to(DEVICE)
    clf_head = nn.Linear(feature_dim, n_classes).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(list(model.parameters()) + list(clf_head.parameters()), lr=lr)

    model.train()
    clf_head.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(DEVICE)
            labels = labels.squeeze().long().to(DEVICE)

            optimizer.zero_grad()
            feats = model(imgs)
            logits = clf_head(feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"[CNN] Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

    return model

def extract_features(model, data_loader):
    model.eval()
    all_feats = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs = imgs.to(DEVICE)
            feats = model(imgs).cpu().numpy()
            all_feats.append(feats)
            all_labels.append(labels.squeeze().numpy())
    X = np.concatenate(all_feats, axis=0)
    y = np.concatenate(all_labels, axis=0)
    return X, y

# PCA for dimensionality reduction
N_QUBITS = 4  # or 6

def apply_pca(train_feats, test_feats, n_components=N_QUBITS):
    pca = PCA(n_components=n_components, random_state=SEED)
    X_train_red = pca.fit_transform(train_feats)
    X_test_red  = pca.transform(test_feats)
    return X_train_red, X_test_red, pca

In [ ]:
def evaluate_classical_models(X_train, y_train, X_test, y_test):
    results = {}

    # Logistic Regression
    logreg = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)
    logreg.fit(X_train, y_train)
    y_pred = logreg.predict(X_test)
    results["LogReg"] = {
        "acc": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, average="weighted"),
    }

    # SVM RBF
    svm_rbf = SVC(kernel="rbf", random_state=SEED)
    svm_rbf.fit(X_train, y_train)
    y_pred = svm_rbf.predict(X_test)
    results["SVM_RBF"] = {
        "acc": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, average="weighted"),
    }

    # Small MLP (capacity-matched)
    mlp = MLPClassifier(hidden_layer_sizes=(32,),
                        activation="relu",
                    max_iter=1000, # Increased from 500 to 1000                        random_state=SEED)
                    random_state=SEED)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    results["MLP"] = {
        "acc": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, average="weighted"),
    }

    return results

In [ ]:
# Trainable Quantum Kernel Implementation
def create_qkernel(n_qubits, depth=1):
    dev = qml.device("lightning.qubit", wires=n_qubits)

    projector = np.zeros((2**n_qubits, 2**n_qubits))
    projector[0, 0] = 1.0

    @qml.qnode(dev, interface="autograd")
    def kernel_qnode(x1, x2, params):
        # Prepare |ψ(x1)>
        for i in range(n_qubits):
            qml.RZ(x1[i], wires=i)
        for d in range(depth):
            for i in range(n_qubits):
                qml.RY(params[d, i], wires=i)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])

        # Apply adjoint of feature map for x2
        for d in reversed(range(depth)):
            for i in reversed(range(n_qubits - 1)):
                qml.CNOT(wires=[i, i + 1])
            for i in reversed(range(n_qubits)):
                qml.RY(-params[d, i], wires=i)
        for i in reversed(range(n_qubits)):
            qml.RZ(-x2[i], wires=i)

        return qml.expval(qml.Hermitian(projector, wires=range(n_qubits)))

    return kernel_qnode, dev

In [ ]:
# Kernel Matrix and Training
def kernel_matrix(X1, X2, params, kernel_fn):
    n1 = X1.shape[0]
    n2 = X2.shape[0]
    K = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            K[i, j] = kernel_fn(X1[i], X2[j], params)
    return K

def rescale_labels_for_kta(y, n_classes):
    y_float = y.astype(float)
    y_centered = y_float - y_float.mean()
    return y_centered

def kta_cost(params, X_sub, y_sub, kernel_fn):
    y_rescaled = rescale_labels_for_kta(y_sub, n_classes)
    K = qml.kernels.square_kernel_matrix(X_sub, lambda x1, x2: kernel_fn(x1, x2, params))
    return -qml.kernels.target_alignment(X_sub, y_rescaled, lambda x1, x2: kernel_fn(x1, x2, params),
                                         assume_normalized_kernel=True)

def train_qkernel(X_train, y_train, kernel_fn, steps=20, lr=0.1, batch_size=64):
    n_qubits = X_train.shape[1]
    params = pnp.random.uniform(low=0.0, high=2*np.pi, size=(1, n_qubits), requires_grad=True)

    opt = qml.GradientDescentOptimizer(lr)

    n = X_train.shape[0]
    idx = np.arange(n)

    for step in range(steps):
        np.random.shuffle(idx)
        batch_idx = idx[:min(batch_size, n)]
        X_sub = X_train[batch_idx]
        y_sub = y_train[batch_idx]

        def cost_fn(p):
            return kta_cost(p, X_sub, y_sub, kernel_fn)

        params, cost_val = opt.step_and_cost(cost_fn, params)
        if (step + 1) % 5 == 0:
            print(f"[QKernel] Step {step+1}/{steps} - KTA cost: {cost_val:.4f}")

    return params

In [ ]:
# SVM with Precomputed Quantum Kernel
def svm_with_qkernel(X_train, y_train, X_test, y_test, kernel_fn, params):
    # Precomputed kernel matrices
    K_train = kernel_matrix(X_train, X_train, params, kernel_fn)
    K_test  = kernel_matrix(X_test,  X_train, params, kernel_fn)

    svm = SVC(kernel="precomputed")
    svm.fit(K_train, y_train)
    y_pred = svm.predict(K_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="weighted")
    return acc, f1

In [ ]:
# Main Experiment Loop
data_fractions = [0.05, 0.10, 0.20]
feature_dim = 64
qkernel_depth = 1
qkernel_steps = 15   # Keep small for Colab runtime

results_table = []

for frac in data_fractions:
    print("\n" + "="*60)
    print(f"Data fraction: {int(frac*100)}% of train set")
    print("="*60)

    # 1) Data loaders
    train_loader = get_loader(train_dataset, batch_size=128, subset_fraction=frac, shuffle=True)
    full_train_loader = get_loader(train_dataset, batch_size=128, subset_fraction=frac, shuffle=False)
    test_loader  = get_loader(test_dataset,  batch_size=128, subset_fraction=None, shuffle=False)

    # 2) Train CNN on subset
    cnn = train_cnn_feature_extractor(train_loader, feature_dim=feature_dim, epochs=5, lr=1e-3)

    # 3) Extract features
    X_train_feat, y_train = extract_features(cnn, full_train_loader)
    X_test_feat,  y_test  = extract_features(cnn, test_loader)

    # 4) PCA -> defines number of qubits
    X_train_red, X_test_red, pca = apply_pca(X_train_feat, X_test_feat, n_components=N_QUBITS)

    print(f"Number of qubits: {N_QUBITS}")
    print(f"CNN feature dim: {feature_dim}")
    print(f"Train samples used: {X_train_red.shape[0]}")

    # 5) Classical baselines
    classical_res = evaluate_classical_models(X_train_red, y_train, X_test_red, y_test)
    for model_name, metrics in classical_res.items():
        results_table.append({
            "data_frac": frac,
            "model": model_name,
            "acc": metrics["acc"],
            "f1": metrics["f1"]
        })
        print(f"{model_name}: acc={metrics['acc']:.4f}, f1={metrics['f1']:.4f}")

    # 6) Quantum kernel (noiseless)
    kernel_fn, dev = create_qkernel(n_qubits=N_QUBITS, depth=qkernel_depth)
    print(f"Quantum device (noiseless): {dev.name}")

    # Train kernel parameters
    qparams = train_qkernel(X_train_red, y_train, kernel_fn, steps=qkernel_steps, lr=0.1, batch_size=64)

    # SVM with quantum kernel
    q_acc, q_f1 = svm_with_qkernel(X_train_red, y_train, X_test_red, y_test, kernel_fn, qparams)
    results_table.append({
        "data_frac": frac,
        "model": "TQK_SVM",
        "acc": q_acc,
        "f1": q_f1
    })
    print(f"TQK + SVM: acc={q_acc:.4f}, f1={q_f1:.4f}")


Data fraction: 5% of train set
[CNN] Epoch 1/5 - Loss: 2.1742
[CNN] Epoch 2/5 - Loss: 1.8971
[CNN] Epoch 3/5 - Loss: 1.5258
[CNN] Epoch 4/5 - Loss: 1.4030
[CNN] Epoch 5/5 - Loss: 1.3220
Number of qubits: 4
CNN feature dim: 64
Train samples used: 4499


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


LogReg: acc=0.6230, f1=0.6217
SVM_RBF: acc=0.6641, f1=0.6460
MLP: acc=0.6185, f1=0.6242
Quantum device (noiseless): lightning.qubit
[QKernel] Step 5/15 - KTA cost: 0.0000
[QKernel] Step 10/15 - KTA cost: -0.0000
[QKernel] Step 15/15 - KTA cost: 0.0000


In [ ]:
# Results Visualization
import pandas as pd

df = pd.DataFrame(results_table)
print("\n=== Performance Table ===")
print(df)

# Plot Accuracy vs data percentage for each model
plt.figure(figsize=(10, 6))
for model_name in df["model"].unique():
    sub = df[df["model"] == model_name]
    x = sub["data_frac"].values * 100
    y = sub["acc"].values
    plt.plot(x, y, marker="o", label=model_name, linewidth=2)

plt.xlabel("Train data percentage (%)", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title(f"{DATA_FLAG.upper()} - Accuracy vs Data Regime", fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print model details
print("\n=== Quantum Circuit Details ===")
print(f"Number of qubits: {N_QUBITS}")
print(f"Trainable parameters (TQK): {N_QUBITS * qkernel_depth}")
print(f"Circuit depth (feature + trainable layers): {1 + qkernel_depth}")
print(f"Entanglement: Linear chain of CNOTs")
print(f"\nNote: The table shows {DATA_FLAG} classification performance")
print("under low-data regimes (5%, 10%, 20% of training data).")
print("Classical baselines: LogReg, SVM-RBF, MLP")
print("Quantum: Trainable Quantum Kernel (TQK) + SVM")